In [18]:
import pandas as pd
import bibtexparser
import glob
import os
import arxiv
import time

def get_arxiv_info(arxiv_id):
    """
    Fetch paper information from arXiv using the arxiv ID.
    Includes rate limiting to be respectful to arXiv's servers.
    
    Args:
        arxiv_id: arXiv identifier (can be with or without version number)
    
    Returns:
        dict: Dictionary with arxiv information or None if not found
    """
    print('arxiv_id: ', arxiv_id)
    try:
        # Remove version number if present (e.g., v1, v2)
        clean_id = arxiv_id.split('v')[0]
        
        # Add rate limiting
        time.sleep(3)  # Be nice to arXiv servers
        
        # Create client and search for the paper
        client = arxiv.Client()
        search = arxiv.Search(id_list=[clean_id])
        paper = next(client.results(search))
        
        # Extract version from paper URL if present
        version = paper.entry_id.split('v')[-1] if 'v' in paper.entry_id else '1'
        
        return {
            'arxiv_url': paper.entry_id,
            'arxiv_primary_category': paper.primary_category,
            'arxiv_version': version,
            'abstract': paper.summary,
            'published_date': paper.published.strftime('%Y-%m-%d') if paper.published else None
        }
    except Exception as e:
        print(f"Error fetching arXiv info for {arxiv_id}: {str(e)}")
        return None

def extract_bib_info(entry, collaboration):
    """
    Extract relevant information from a BibTeX entry.
    
    Args:
        entry: BibTeX entry dictionary
        collaboration: Name of collaboration extracted from filename
    
    Returns:
        dict: Dictionary with extracted information
    """
    # Initialize with default values
    info = {
        'collaboration': collaboration,
        'title': '',
        'bibtag': '',
        'eprint': '',
        'year': '',
        'interaction': None,
        'target': None,
        'arxiv_url': None,
        'arxiv_primary_category': None,
        'arxiv_version': None,
        'abstract': None,
        'authors': None,
        'published_date': None
    }
    
    # Extract bibtag (ID/key of the entry)
    info['bibtag'] = entry.get('ID', '')
    
    # Extract title (remove curly braces if present)
    title = entry.get('title', '')
    info['title'] = title.replace('{', '').replace('}', '')
    
    # Extract year and ensure it's numeric
    year = entry.get('year', '')
    try:
        info['year'] = int(year) if year else None
    except ValueError:
        info['year'] = None
    
    # Extract eprint (arxiv number)
    info['eprint'] = entry.get('eprint', '')
    
    # # If we have an eprint, fetch arXiv information
    # if info['eprint']:
    #     arxiv_info = get_arxiv_info(info['eprint'])
    #     if arxiv_info:
    #         info.update(arxiv_info)
    
    return info

def process_bib_files(directory_path='*.bib'):
    """
    Process all .bib files in the specified directory and create a DataFrame.
    Fetches additional information from arXiv when available.
    
    Args:
        directory_path: Path pattern for .bib files (default: '*.bib' in current directory)
    
    Returns:
        pandas.DataFrame: DataFrame containing information from all .bib files and arXiv
    """
    all_entries = []
    
    # Find all .bib files
    bib_files = glob.glob(directory_path)

    cnt=0
    for bib_file in bib_files:
        # Extract collaboration name from filename (remove .bib extension)
        collaboration = os.path.splitext(os.path.basename(bib_file))[0]
        
        try:
            # Read the .bib file
            with open(bib_file, 'r', encoding='utf-8') as bibtex_file:
                parser = bibtexparser.bparser.BibTexParser()
                bib_database = bibtexparser.load(bibtex_file, parser=parser)
            
            # Process each entry in the file
            for entry in bib_database.entries:
                entry_info = extract_bib_info(entry, collaboration)
                all_entries.append(entry_info)
                
        except Exception as e:
            print(f"Error processing {bib_file}: {str(e)}")
    
    # Create DataFrame
    df = pd.DataFrame(all_entries)
    
    # Ensure all required columns are present
    required_columns = ['collaboration', 'title', 'bibtag', 'eprint', 'year',
                       'interaction', 'target', 'arxiv_url', 'arxiv_primary_category',
                       'arxiv_version', 'abstract', 'authors', 'published_date']
    
    for col in required_columns:
        if col not in df.columns:
            df[col] = None
    
    # Sort by collaboration and year (descending) and reset index
    df = df[required_columns].sort_values(['collaboration', 'year'], 
                                        ascending=[True, False],
                                        na_position='last').reset_index(drop=True)
    
    return df

In [19]:
df = process_bib_files('*.bib')

df.to_csv('bibliography_database.csv', index=False)

In [20]:
df = pd.read_csv('bibliography_database.csv')

In [21]:
df

,collaboration,title,bibtag,eprint,year,interaction,target,arxiv_url,arxiv_primary_category,arxiv_version,abstract,authors,published_date
0,ArgoNeuT,First measurement of electron neutrino scatter...,ArgoNeuT:2020kir,2004.01956,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ArgoNeuT,First measurement of the cross section for $\n...,ArgoNeuT:2018und,1804.10294,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ArgoNeuT,Measurement of $\nu_\mu$ and $\bar\nu_\mu$ neu...,ArgoNeuT:2015ldo,1511.00941,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ArgoNeuT,Detection of Back-to-Back Proton Pairs in Char...,ArgoNeuT:2014ihi,1405.4261,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ArgoNeuT,First Measurement of Neutrino and Antineutrino...,ArgoNeuT:2014uwh,1408.0598,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,T2K,Measurement of the electron neutrino charged-c...,T2K:2015ydf,1503.08815,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138,T2K,Measurement of the Inclusive Electron Neutrino...,T2K:2014lbi,1407.7389,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
139,T2K,Measurement of the inclusive $\nu_\mu$ charged...,T2K:2014axs,1407.4256,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140,T2K,Measurement of the neutrino-oxygen neutral-cur...,T2K:2014vog,1403.3140,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
